# CT-RATE Temporal Labeling v5 — VALIDATED per-finding temporal_sentence

Same 18-finding, text-precedence pipeline and resumable Drive writes as v4, plus a
deterministic semantic gate that rejects static current-study descriptions masquerading
as temporal comparisons. **One validated field per finding:**
`temporal_sentence` = the full verbatim sentence from the CURRENT report stating that
finding's change vs prior (empty when `state == not_mentioned`).

This gives the supervised temporal-contrastive loss a clean, single-finding,
single-direction sentence, so it can hold finding identity constant and contrast only
temporal direction.

**Before running:**
1. Runtime > Change runtime type > A100 GPU (80GB / High-RAM).
2. Accept the license: https://huggingface.co/google/medgemma-27b-text-it
3. Upload `ctrate_pairs_enriched_v2.csv` (from scripts/16_join_abnormality_labels.py).

In [ ]:
# 1. Deps
!pip -q install -U 'transformers>=4.50' accelerate huggingface_hub

In [ ]:
# 2. GPU sanity — expect an A100 with ~80 GB
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > A100 GPU'
p = torch.cuda.get_device_properties(0)
print('GPU:', p.name, f'{p.total_memory/1e9:.0f} GB  torch', torch.__version__)
if p.total_memory/1e9 < 70:
    print('WARNING: <70 GB. bf16 27B needs ~54 GB weights + activations; pick the 80 GB A100.')

In [ ]:
# 3. Hugging Face login (gated MedGemma weights)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 4. Upload ctrate_pairs_enriched_v2.csv
from google.colab import files
import csv
csv.field_size_limit(10**9)
up = files.upload()
MANIFEST = list(up.keys())[0]
with open(MANIFEST, newline='', encoding='utf-8') as f:
    ROWS = list(csv.DictReader(f))
print(f'{len(ROWS)} pairs loaded from {MANIFEST}')
need = ['presence_changes', 'prior_labels', 'curr_labels']
missing = [c for c in need if c not in ROWS[0]]
assert not missing, f'Missing {missing}. Upload ctrate_pairs_enriched_v2.csv (run scripts/16 first).'
print('v2 manifest OK.')

In [ ]:
# 5. v5 prompt + deterministic temporal gate; TEXT PRECEDENCE merge
import json, re
NL = chr(10)

CANON = ['Medical material','Arterial wall calcification','Cardiomegaly',
  'Pericardial effusion','Coronary artery wall calcification','Hiatal hernia',
  'Lymphadenopathy','Emphysema','Atelectasis','Lung nodule','Lung opacity',
  'Pulmonary fibrotic sequela','Pleural effusion','Mosaic attenuation pattern',
  'Peribronchial thickening','Consolidation','Bronchiectasis','Interlobular septal thickening']

SYSTEM_PROMPT = (
  'You are an expert thoracic radiologist. You are given the PRIOR and CURRENT CT reports '
  'for the same patient. For each finding in a fixed list, report ONLY what the CURRENT '
  'report explicitly says about how it CHANGED relative to the prior study. '
  'NEVER compare the two reports yourself to derive a change. A state is explicit ONLY when '
  'the CURRENT report sentence itself uses temporal/comparative wording. '
  'Do not infer change from absence of mention. If the current report does not explicitly compare '
  'that finding to the prior, say not_mentioned. Respond with ONE JSON object.')

SCHEMA = NL.join([
  'Return ONLY a JSON object:',
  '{"findings": [{"finding": <exact name from the FINDINGS list>, "state": one of ["up","down","same","not_mentioned"], "evidence": <short quote from CURRENT report, <=15 words, or "">, "temporal_sentence": <the FULL sentence copied verbatim from the CURRENT report that states this finding change vs prior, or "">}]}',
  'Definitions:',
  '- "up"   = CURRENT sentence itself says larger / more / increased / progressed / newly appeared / gives a prior measurement.',
  '- "down" = CURRENT sentence itself says smaller / fewer / decreased / regressed / resolved / no longer seen / gives a prior measurement.',
  '- "same" = CURRENT report EXPLICITLY says unchanged / stable / persistent / no significant change.',
  '- "not_mentioned" = current report states no comparison for this finding. USE WHENEVER IN DOUBT.',
  'Rules:',
  '- Include exactly one entry for EVERY finding in the FINDINGS list, in that order.',
  '- "evidence" must be copied verbatim from the CURRENT report; leave "" for not_mentioned.',
  '- "temporal_sentence" must be a COMPLETE sentence copied verbatim from the CURRENT report that expresses this finding change or stability; it should CONTAIN the evidence quote; leave "" when not_mentioned.',
  '- The sentence must contain an explicit cue such as interval, prior, previous(ly), compared, since, new/newly, increased/decreased, larger/smaller, improved/worsened, resolved, unchanged, stable, remains, or a current-vs-prior measurement.',
  '- NEVER infer change by contrasting a static PRIOR sentence with a static CURRENT sentence.',
  '- Static examples such as "There is pleural effusion", "Heart size is normal", "A catheter is seen", or "Atelectasis is present" are NOT temporal, even if the prior report differs -> not_mentioned.',
  '- Merely describing a finding is NOT a comparison -> not_mentioned (temporal_sentence "").',
  '- Output ONLY the JSON object. No prose, no markdown fences.'])

EXAMPLES = NL.join([
  'Worked example (abbreviated).',
  'PRIOR REPORT: Heart size is increased. No pleural effusion. Subcarinal lymph node 9 mm.',
  'CURRENT REPORT: Heart contour and size are normal. There is bilateral pleural effusion. The subcarinal lymph node short axis is 15 mm, previously 9 mm. Millimetric nodules are noted in both lungs.',
  'CORRECT OUTPUT (relevant entries): {"findings":[{"finding":"Cardiomegaly","state":"not_mentioned","evidence":"","temporal_sentence":""},{"finding":"Pleural effusion","state":"not_mentioned","evidence":"","temporal_sentence":""},{"finding":"Lymphadenopathy","state":"up","evidence":"short axis is 15 mm, previously 9 mm","temporal_sentence":"The subcarinal lymph node short axis is 15 mm, previously 9 mm."},{"finding":"Lung nodule","state":"not_mentioned","evidence":"","temporal_sentence":""}]}',
  'Why: normal heart size and bilateral effusion differ from the PRIOR report, but their CURRENT sentences contain no comparison; do not infer down/up. The node sentence explicitly says previously 9 mm, so it is temporal.'])

def curr_text(row):
    return (row.get('curr_findings','') + ' ' + row.get('curr_impression','')).strip()

def build_user(row):
    prior = (row.get('prior_findings','') + ' ' + row.get('prior_impression','')).strip()
    parts = [EXAMPLES, '', 'Now do this case.',
             'FINDINGS list (one entry each, this order): ' + '; '.join(CANON),
             'INTERVAL between studies: ' + str(row.get('delta_days','')) + ' days', '',
             'PRIOR REPORT:', prior or '(none)', '',
             'CURRENT REPORT:', curr_text(row), '', SCHEMA]
    return NL.join(parts)

def extract_json(text):
    text = text.replace('```json','').replace('```','')
    i = text.find('{')
    if i < 0: return None
    d = 0
    for j in range(i, len(text)):
        if text[j] == '{': d += 1
        elif text[j] == '}':
            d -= 1
            if d == 0:
                try: return json.loads(text[i:j+1])
                except Exception: return None
    return None

STATE_TO_CHANGE = {'up': 'worse', 'down': 'improved', 'same': 'stable'}
DIRECTION = {'new':'worsened','worse':'worsened','stable':'stable','improved':'improved','resolved':'improved'}

TEMPORAL_CUE = re.compile(
    r'\b(interval|prior|previous|previously|compared|comparison|since|formerly|as before|again|'
    r'unchanged|stable|stability|remains|remained|still|persistent|persisting|similar|'
    r'new|newly|developed|appeared|no longer|disappeared|resolved|resolution|'
    r'increas\w*|decreas\w*|enlarg\w*|larger|smaller|worsen\w*|improv\w*|'
    r'progress\w*|regress\w*|reduc\w*|grew|growth|change\w*)\b', re.I)
UP_CUE = re.compile(
    r'\b(new|newly|developed|appeared|increas\w*|enlarg\w*|larger|worsen\w*|'
    r'progress\w*|grew|growth)\b', re.I)
DOWN_CUE = re.compile(
    r'\b(no longer|disappeared|resolved|resolution|decreas\w*|smaller|improv\w*|'
    r'regress\w*|reduc\w*)\b', re.I)
REFERENCE_CUE = re.compile(r'\b(previous|previously|prior|compared|since)\b', re.I)
SAME_CUE = re.compile(
    r'\b(unchanged|stable|stability|no significant (interval )?change|remains? unchanged|'
    r'persistent|persisting|remains? stable|similar to (the )?(prior|previous)|as before)\b', re.I)

def norm_text(s):
    return ' '.join((s or '').split()).lower().strip()

def validate_explicit(row, st, ev, ts):
    if st not in ('up','down','same'):
        return False, 'not_explicit_state'
    if not ts.strip():
        return False, 'empty_temporal_sentence'
    current = norm_text(curr_text(row)); sent = norm_text(ts); quote = norm_text(ev)
    if sent not in current:
        return False, 'sentence_not_verbatim'
    if quote and quote not in current:
        return False, 'evidence_not_verbatim'
    if quote and quote not in sent:
        return False, 'evidence_not_in_sentence'
    if not TEMPORAL_CUE.search(ts):
        return False, 'no_temporal_cue'
    has_up, has_down, has_ref = bool(UP_CUE.search(ts)), bool(DOWN_CUE.search(ts)), bool(REFERENCE_CUE.search(ts))
    direction_ok = ((st == 'up' and not has_down and (has_up or has_ref)) or
                    (st == 'down' and not has_up and (has_down or has_ref)) or
                    (st == 'same' and not has_up and not has_down and SAME_CUE.search(ts)))
    if not direction_ok:
        return False, 'cue_state_mismatch'
    return True, 'ok'

def combine(row, parsed):
    try:
        pres = json.loads(row.get('presence_changes','') or '{}')
    except Exception:
        pres = {}
    states = {}
    if parsed:
        for e in (parsed.get('findings') or []):
            fn, st = e.get('finding'), e.get('state')
            if fn in CANON and st in ('up','down','same','not_mentioned'):
                ev = (e.get('evidence','') or '')[:160]
                ts = (e.get('temporal_sentence','') or '').strip()[:400]
                valid, reason = validate_explicit(row, st, ev, ts)
                states[fn] = (st, ev, ts, valid, reason)
    out = []
    for f in CANON:
        raw_st, raw_ev, raw_ts, valid, reason = states.get(
            f, ('not_mentioned', '', '', False, 'missing_model_entry'))
        st, ev, ts = (raw_st, raw_ev, raw_ts) if valid else ('not_mentioned', '', '')
        p = pres.get(f)
        if st in ('up','down','same'):
            change = STATE_TO_CHANGE[st]
            if st == 'up' and p == 'new':
                change = 'new'
            elif st == 'down' and p == 'resolved':
                change = 'resolved'
            out.append({'finding': f, 'change': change, 'direction': DIRECTION[change],
                        'tier': 'explicit', 'state': st, 'presence': p or 'absent_both',
                        'agrees_with_presence': bool(
                            (st=='up' and p in ('new','present_both')) or
                            (st=='down' and p in ('resolved','present_both')) or
                            (st=='same' and p == 'present_both')),
                        'evidence': ev, 'temporal_sentence': ts,
                        'temporal_validated': True, 'rejection_reason': None})
        else:
            if p == 'new':
                change = 'new'
            elif p == 'resolved':
                change = 'resolved'
            elif p == 'present_both':
                change = 'no_comparison'
            else:
                continue
            out.append({'finding': f, 'change': change,
                        'direction': DIRECTION.get(change, 'unknown'),
                        'tier': 'inferred', 'state': 'not_mentioned',
                        'presence': p, 'agrees_with_presence': None,
                        'evidence': '', 'temporal_sentence': '',
                        'temporal_validated': False,
                        'rejected_state': raw_st if raw_st in ('up','down','same') else None,
                        'rejection_reason': reason if raw_st in ('up','down','same') else None,
                        'rejected_evidence': raw_ev if raw_st in ('up','down','same') else '',
                        'rejected_temporal_sentence': raw_ts if raw_st in ('up','down','same') else ''})
    return out
print('v5 helpers ready — verbatim + temporal-cue gate enabled')

In [ ]:
# 6. Load MedGemma-27B in bf16 straight onto the GPU
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, time, gc, os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
for _v in ['model', '_o', '_e', 'out', 'enc']:
    if _v in globals():
        try: del globals()[_v]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'GPU free before load: {free_gb:.1f} GB')
assert free_gb > 60, ('Only %.1f GB free -> a previous model is still resident. '
                      'Runtime > Restart session, then run cells 1-6 again.' % free_gb)
MODEL_ID = 'google/medgemma-27b-text-it'
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = 'left'
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, attn_implementation='sdpa').to('cuda')
model.config.use_cache = True
model.generation_config.use_cache = True
model.eval()
print('model loaded, bf16, use_cache =', model.config.use_cache)

_p = tok.apply_chat_template([{'role':'user','content':'Reply with the single word: ok'}],
                             tokenize=False, add_generation_prompt=True)
_e = tok(_p, return_tensors='pt').to('cuda')
torch.cuda.synchronize(); _t0 = time.time()
with torch.inference_mode():
    _o = model.generate(**_e, max_new_tokens=64, do_sample=False, use_cache=True,
                        pad_token_id=tok.pad_token_id)
torch.cuda.synchronize(); _dt = time.time() - _t0
_ntok = _o.shape[1] - _e['input_ids'].shape[1]
print(f'WARMUP: {_ntok} tokens in {_dt:.1f}s = {_ntok/max(_dt,1e-6):.1f} tok/s')
print('  -> expect >20 tok/s on the A100.')

In [ ]:
# 7. Labeling — writes STRAIGHT TO DRIVE, resumable
from google.colab import drive
import os
NL = chr(10)
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/ct_temporal'
os.makedirs(DRIVE_DIR, exist_ok=True)
OUT = os.path.join(DRIVE_DIR, 'medgemma_labels_v5.jsonl')  # NEW file: never resume flawed v4 pilot
LIMIT = 50    # pilot first. Set 0 for all pairs once QC looks right.
BATCH = 8
MAX_NEW = 2000
RESUME = True

rows_all = ROWS if LIMIT == 0 else ROWS[:LIMIT]
done = set()
if RESUME and os.path.exists(OUT):
    for l in open(OUT):
        try:
            d = json.loads(l); done.add((d['prior_volume'], d['curr_volume']))
        except Exception: pass
rows = [r for r in rows_all if (r['prior_volume'], r['curr_volume']) not in done]
print(f'{len(rows_all)} selected, {len(done)} already done, {len(rows)} to label this run')

prompts = [tok.apply_chat_template(
               [{'role':'system','content':SYSTEM_PROMPT},
                {'role':'user','content':build_user(r)}],
               tokenize=False, add_generation_prompt=True)
           for r in rows]

MAXLEN = 8192
tok.truncation_side = 'left'
if prompts:
    lens = sorted(len(tok(p)['input_ids']) for p in prompts)
    print(f'prompt tokens: median={lens[len(lens)//2]} max={lens[-1]} (cap={MAXLEN})')

fout = open(OUT, 'a' if (RESUME and done) else 'w', encoding='utf-8')
n_ok = n_bad = 0
n_batches = (len(prompts) + BATCH - 1) // BATCH
for bi, i in enumerate(range(0, len(prompts), BATCH), 1):
    bp, br = prompts[i:i+BATCH], rows[i:i+BATCH]
    if not bp: break
    enc = tok(bp, return_tensors='pt', padding=True, truncation=True, max_length=MAXLEN).to('cuda')
    print(f'  batch {bi}/{n_batches} generating...', flush=True)
    torch.cuda.synchronize(); t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                             use_cache=True, pad_token_id=tok.pad_token_id)
    torch.cuda.synchronize(); dt = time.time() - t0
    texts = tok.batch_decode(out[:, enc['input_ids'].shape[1]:], skip_special_tokens=True)
    for row, raw in zip(br, texts):
        parsed = extract_json(raw)
        findings = combine(row, parsed)
        if parsed is not None: n_ok += 1
        else: n_bad += 1
        rec = {'patient': row['patient'], 'prior_volume': row['prior_volume'],
               'curr_volume': row['curr_volume'], 'delta_days': row['delta_days'],
               'findings': findings, 'parse_ok': parsed is not None}
        if parsed is None: rec['raw'] = raw[:2000]
        fout.write(json.dumps(rec) + NL)
    fout.flush()
    print(f'    {dt:.1f}s  ({i+len(bp)}/{len(prompts)} this run)', flush=True)
fout.close()
print(f'DONE. parse_ok={n_ok} fail={n_bad} -> {OUT} (total lines {sum(1 for _ in open(OUT))})')

In [ ]:
# 8. QC — semantic gate, coverage, and verbatim check
from collections import Counter

def norm(s):
    return ' '.join((s or '').split()).lower()

CURR = {(r['prior_volume'], r['curr_volume']): norm(curr_text(r)) for r in ROWS}
recs = [json.loads(l) for l in open(OUT) if l.strip()]
ok = sum(r['parse_ok'] for r in recs)
print(f'records={len(recs)}  parse_ok={ok} ({100*ok/max(len(recs),1):.1f}%)')

tier = Counter(); direction = Counter(); rejected = Counter()
ts_by_dir = Counter(); ts_have_by_dir = Counter()
ts_ok = ts_bad = 0
bad = []; rejected_examples = []
for r in recs:
    ctext = CURR.get((r['prior_volume'], r['curr_volume']), '')
    for fd in r['findings']:
        tier[fd['tier']] += 1
        direction[fd['direction']] += 1
        if fd.get('rejection_reason'):
            rejected[fd['rejection_reason']] += 1
            if len(rejected_examples) < 12:
                rejected_examples.append((fd['finding'], fd.get('rejected_state'), fd['rejection_reason'],
                                          (fd.get('rejected_temporal_sentence') or '')[:100]))
        if fd['tier'] != 'explicit':
            continue
        d = fd['direction']
        ts_by_dir[d] += 1
        ts = (fd.get('temporal_sentence','') or '').strip()
        if ts:
            ts_have_by_dir[d] += 1
            if ctext and norm(ts) in ctext:
                ts_ok += 1
            else:
                ts_bad += 1
                if len(bad) < 8:
                    bad.append((fd['finding'], d, ts[:100]))

print('tier        :', dict(tier))
print('3-class dir :', dict(direction))
print('rejected false-explicit candidates:', dict(rejected))
for fn, st, why, sent in rejected_examples:
    print('  REJECTED:', st, '|', fn, '|', why, '|', sent)
print('--- temporal_sentence coverage among EXPLICIT findings ---')
for d in ['worsened','stable','improved']:
    n = ts_by_dir[d]; h = ts_have_by_dir[d]
    print(f'  {d:9}: {h}/{n} have a temporal_sentence ({100*h/max(n,1):.1f}%)')
tot_ts = ts_ok + ts_bad
print(f'verbatim check: {ts_ok}/{tot_ts} found in current report ({100*ts_ok/max(tot_ts,1):.1f}%)')
for fn, d, s in bad:
    print('  NON-verbatim:', d, '|', fn, '|', s)

print('--- spot-check (first 2 pairs, explicit only) ---')
for r in recs[:2]:
    print(r['patient'], str(r['delta_days']) + 'd')
    for fd in r['findings']:
        if fd['tier'] == 'explicit':
            print('  ', fd['finding'], fd['change'], '[', fd['state'], ']')
            print('     ts:', (fd.get('temporal_sentence') or '')[:100])

In [ ]:
# 9. Already on Drive; also pull a local backup
from google.colab import files
import os
print('Drive copy:', OUT, '| exists:', os.path.exists(OUT),
      '| lines:', (sum(1 for _ in open(OUT)) if os.path.exists(OUT) else 0))
files.download(OUT)

## Pilot QC checklist

1. `parse_ok` high (>95%).
2. Inspect `rejected false-explicit candidates`: static descriptions should be rejected mainly as
   `no_temporal_cue`; paraphrases as `sentence_not_verbatim`.
3. Every surviving explicit example must have `temporal_validated=True` and a real comparison cue.
4. Manually inspect at least 20 surviving temporal sentences; `not_mentioned`/inferred should dominate.

Then set `LIMIT = 0` and run all pairs. Output persists to Drive (`medgemma_labels_v5.jsonl`)
and resumes on disconnect.

Note on presence vs severity: `new`/`resolved` findings often carry a presence sentence rather
than an increase/decrease comparison; directional language concentrates in `present_both`
severity findings. The training pipeline can run the contrastive term on severity findings only,
or treat new/resolved as worsened/improved positives — a training-side flag, decided separately.